# Lecture 02, Notebook 06: Lux Training Fundamentals

The Python notebook introduces PyTorch modules, tensors, gradients, and
optimizers. Here the same concepts are shown with Lux's explicit
`model(x, ps, st)` interface and the shared training helper.

Read this after `lecture_02_00_Lux_orientation.ipynb` if you want the
mechanics behind later notebooks. The goal is to see exactly where model
parameters `ps`, threaded layer state `st`, gradients, optimizer state, and
metrics live.

## Lecture 02, Notebook 06: PyTorch Fundamentals

**Course:** Deep Learning for Solving and Estimating Dynamic Models in Economics and Finance
**Script reference:** §1.5–1.8 (Optimization and training: PyTorch fundamentals)
**Notebook role:** core
**Author:** Simon Scheidegger

*Julia/Lux preview of* `lectures/lecture_02_intro_deep_learning/code/lecture_02_06_PyTorch_intro.ipynb`.

> **Run mode.** The checked-in run uses `RUN_MODE = "smoke"` for fast execution; set `RUN_MODE` to `"teaching"` or `"production"` in the setup cell below for the longer training budgets.

## From PyTorch modules to Lux training fundamentals

The Python ground-truth notebook is a PyTorch tour: it builds `nn.Module` networks and trains them with `torch.optim` on two tasks — a **regression** example (MSE loss) and a binary **classification** example (BCE loss). PyTorch bundles parameters, layer state, and gradients inside the module object.

This Julia preview teaches the **same training fundamentals in Lux**, where those pieces stay explicit instead of hidden inside an object:

- the **model** is a pure description of the architecture (`make_mlp`), separate from
- its **parameters** `ps` and layer **state** `st`, and
- **gradients** come from `Zygote`, while **optimizer state** lives in an `Optimisers.jl` rule.

We walk the full loop on a supervised **regression** target: define the model, inspect a single gradient with `Zygote.pullback`, run the shared `train_step!` helper, and visualise the fit. The Python notebook's classification example (BCE loss, sigmoid output, decision-boundary plot) is summarised near the end — the training mechanics are identical; only the loss and the data differ.

Lux separates the model architecture from its parameters and state. A model call
returns both predictions and updated state:

```julia
prediction, st_new = model(x, ps, st)
```

That is the pattern used throughout the Julia translations, including the DEQN
and PINN residual notebooks.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, "..", "..", "..", "julia"))

using CairoMakie
using DLEFJulia
using Lux
using NNlib
using Optimisers
using Statistics
using Zygote

In [ ]:
RUN_MODE = "smoke"
SEED = 0
budgets = (
    smoke = (epochs = 30, batch_size = 32),
    teaching = (epochs = 200, batch_size = 64),
    production = (epochs = 1_000, batch_size = 128),
)
hp = run_mode_budget(RUN_MODE; budgets)
rng = rng_from_seed(SEED)

---
### 1. Regression example with a neural network

The Python notebook fits a small MLP to noisy data from $y = 2x + 1 + \text{noise}$. Here we use a smooth nonlinear target so the training mechanics are visible on a slightly harder fit; the Lux boundary is the point, not the exact curve.

We build the model with `make_mlp` (input dimension 1, two hidden layers, `tanh` activation, output dimension 1) and pair it with an `Optimisers.Adam` rule via `setup_training`. The inputs are already in Lux's **feature-by-batch** layout — one feature row, many observation columns — so the explicit call is `prediction, st_new = model(batch.x, ps, st)`.

In [ ]:
x = reshape(collect(range(-2.0, 2.0; length = 160)), 1, :)
y = @. 0.5 * x^3 - x + 0.15 * sin(8x)
model = make_mlp(1, (20, 20), 1; activation = NNlib.tanh)
state = setup_training(rng_from_seed(SEED; offset = 1), model, Optimisers.Adam(0.01); parameter_type = Float64)

supervised_loss(model, ps, st, batch) = begin
    prediction, st_new = model(batch.x, ps, st)
    return mse_loss(prediction, batch.y), st_new
end
full_batch = (x = x, y = y)

#### Inspecting a gradient

PyTorch computes gradients implicitly when you call `loss.backward()`. In Lux the gradient is an explicit object: `Zygote.pullback` evaluates the scalar loss and returns a function that backpropagates through the parameters `ps`, leaving Lux state handling visible. This is exactly what the shared `train_step!` helper automates in the next cell.

In [ ]:
initial_loss = loss_value(state, supervised_loss, full_batch)
(loss_before, st_candidate), back = Zygote.pullback(state.ps) do ps
    supervised_loss(state.model, ps, state.st, full_batch)
end
manual_grads = only(back((one(loss_before), nothing)))
manual_grad_norm = sqrt(tree_sum_abs2(manual_grads))

#### Training the regression model

The Python notebook trains with `nn.MSELoss()` and `optim.Adam`. The Lux equivalent is the shared `train_step!` helper, which repeats the same pieces on each step: compute the loss and updated state, take parameter gradients with `Zygote`, clip the gradient norm, update the `Optimisers.Adam` state, and record finite metrics. Later economic notebooks plug residual losses into this same shape.

In [ ]:
history = NamedTuple[]
for _ in 1:hp.epochs
    metrics = train_step!(state, supervised_loss, full_batch; max_grad_norm = 10.0)
    append_metric!(history; step = metrics.step, loss = metrics.loss, grad_norm = metrics.grad_norm)
end
final_loss = loss_value(state, supervised_loss, full_batch)

#### Visualizing the regression results

We compare the trained Lux MLP's predictions to the true target across the input range.

The full Python notebook also plots the **regression training-loss curve** (loss vs epoch) to visualise convergence. In this preview that curve is not drawn; the per-step `history` of `loss` and `grad_norm` is recorded in the training cell above and surfaced numerically through `initial_loss`, `final_loss`, and the step count in the closing summary.

In [ ]:
prediction, _ = state.model(x, state.ps, state.st)
fig = Figure(size = figure_size(RUN_MODE))
ax = Axis(fig[1, 1], xlabel = "x", ylabel = "y")
lines!(ax, vec(x), vec(y); label = "target", color = :black, linewidth = 3)
lines!(ax, vec(x), vec(prediction); label = "Lux MLP", color = :dodgerblue3, linewidth = 3)
axislegend(ax; position = :lt)
fig

---
### The full Python notebook also covers: classification

The PyTorch ground truth continues with a second task — binary **classification**:

- **Data.** Two clusters drawn from distinct Gaussian distributions, giving a 2-feature, 2-class problem.
- **Model.** A small MLP (input dimension 2, one hidden layer, sigmoid output) returning class probabilities.
- **Loss.** Binary cross-entropy, $J = -[y\log\hat{y} + (1-y)\log(1-\hat{y})]$, minimised with Adam — the same optimizer as the regression task.
- **Output.** The decision boundary is visualised by predicting class probabilities over a grid of the feature space.

In Lux this is the identical training loop shown above: only the final activation (sigmoid), the loss kernel (BCE instead of MSE), and the data change. The `model, ps, st` / `Zygote` / `Optimisers` machinery is unchanged.

---
### Conclusion

This notebook re-cast the PyTorch introduction as **Lux training fundamentals**. Using a supervised regression example we saw exactly where each piece lives:

- the **model** (`make_mlp`) is separate from its **parameters** `ps` and **state** `st`,
- **gradients** come from `Zygote.pullback`,
- **optimizer state** lives in an `Optimisers.Adam` rule driven by `train_step!`.

The Python ground truth reuses this same loop for a binary-classification MLP (BCE loss, sigmoid output). Together, regression and classification are the foundation for the deeper architectures and economic residual losses that follow.

**Next steps.**

- `lecture_02_07_Genz_Approximation_and_Loss_Functions_Lux.ipynb` shows the shared loss kernels used later in stochastic residuals.
- `../../lecture_03_deep_equilibrium_nets/code_julia/lecture_03_01_Brock_Mirman_1972_DEQN_Lux.ipynb` is the first notebook where this training pattern drives an economic equilibrium residual.

The cell below returns a machine-checkable summary of this notebook's run.

In [ ]:
(
    initial_loss = initial_loss,
    final_loss = final_loss,
    manual_pullback_loss = loss_before,
    manual_grad_norm = manual_grad_norm,
    candidate_state_type = typeof(st_candidate),
    steps = length(history),
)